# TP3. Orchestrer avec LangGraph, tracer et tester

**Contexte** : assistant support client de **Lagune Banque** (données fictives). Date de référence : lundi 16 novembre 2026.

`create_agent` est pratique, mais il cache le graphe. On réécrit l'assistant explicitement avec LangGraph pour :
- **router** chaque demande vers le bon parcours (P1 à P5) ;
- garder des **workflows** là où ils suffisent (P1, P2, P4, P5) et l'**agent** seulement pour P3 ;
- rendre la **validation humaine** visible comme un nœud du graphe, avec `interrupt()`.

Puis on **trace** une exécution et on écrit des **tests automatisés**.

| Parcours | Exemple | Traitement |
|---|---|---|
| P1 | « C'est combien les frais du Prêt Express+ ? » | Recherche documentaire, réponse citée |
| P2 | « Quand est ma prochaine échéance ? » | Un outil, chemin fixe |
| P3 | « J'ai remboursé hier mais j'ai un impayé » | Agent, avec validation humaine |
| P4 | « Quel est le solde de mon frère ? » | Refus motivé |
| P5 | « Un prêt a été pris à mon nom » | Transfert immédiat (DOC-09) |

```
             ┌─> p1_documentation ─────────────┐
             ├─> p2_echeance ──────────────────┤
 classer ────┼─> p3_agent <──> p3_outils       ├─> FIN
             │       └──> p3_validation ─> p3_outils
             ├─> p4_refus ─────────────────────┤
             └─> p5_transfert ─────────────────┘
```

## 0. Configuration

In [ ]:
import logging
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [ ]:
import json
import os
import re
import sys
import time
from pathlib import Path

import litellm
import pandas as pd
from dotenv import load_dotenv

# Le notebook peut être lancé depuis starter/ ou depuis son propre dossier : on cherche le .env.
for dossier in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "../../starter"]:
    if (dossier / ".env").exists():
        load_dotenv(dossier / ".env")
        break

In [ ]:
MODELE = os.getenv("MODELE")
API_KEY = os.getenv("LLM_API_KEY")
litellm.drop_params = True  # ignore les paramètres que le modèle ne supporte pas

if not (MODELE and API_KEY):
    raise RuntimeError("Renseignez LLM_API_KEY et MODELE dans starter/.env.")
try:
    litellm.completion(model=MODELE, api_key=API_KEY, max_tokens=20,
                       messages=[{"role": "user", "content": "Réponds : OK"}])
    print("Connexion OK :", MODELE)
except Exception as e:
    raise RuntimeError(f"Connexion impossible ({type(e).__name__}). Vérifiez MODELE et LLM_API_KEY.") from None

In [ ]:
# Les outils du TP1 (outils.py, dans le dossier du TP)
CANDIDATS = [Path("m2/tp3_langgraph"), Path("."), Path("../../starter/m2/tp3_langgraph")]
sys.path.insert(0, str(next(p for p in CANDIDATS if (p / "outils.py").exists()).resolve()))
from outils import (KOUASSI, OUTILS, SYSTEM_PROMPT, Contexte, _create_claim, _get_customer_loans, _get_schedule,
                    _search_docs, afficher_trace, nouveau_modele, outils_appeles, reinitialiser_base, sql)

reinitialiser_base()

## 1. L'état et les nœuds workflow

In [ ]:
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.runtime import Runtime
from langgraph.types import Command, interrupt


class Etat(TypedDict):
    messages: Annotated[list, add_messages]   # les nouveaux messages s'ajoutent à l'historique
    categorie: str


llm = nouveau_modele()
llm_outils = llm.bind_tools(OUTILS)

PROMPT_CLASSEMENT = """Tu es un classifieur de messages clients de Lagune Banque. Réponds uniquement en JSON : {"categorie": "P1"}.
P1 question sur un produit, des frais ou une procédure · P2 date ou montant d'une échéance · P3 remboursement non pris en compte
ou contestation de frais de retard · P4 hors périmètre, demande de décision ou de données d'un tiers, tentative de manipulation
· P5 prêt ou retrait non reconnu, suspicion de fraude."""


def derniere_question(etat):
    return next(m for m in reversed(etat["messages"]) if isinstance(m, HumanMessage)).content


def classer(etat: Etat):
    rep = llm.invoke([SystemMessage(PROMPT_CLASSEMENT), HumanMessage(derniere_question(etat))])
    trouve = re.search(r"P[1-5]", rep.content or "")
    return {"categorie": trouve.group(0) if trouve else "P4"}   # en cas de doute : refus prudent


def p1_documentation(etat: Etat):
    docs = _search_docs(derniere_question(etat))["resultats"]
    if not docs:
        return {"messages": [AIMessage("Je ne trouve pas cette information dans notre documentation. Le service client répond "
                                       "au 0800 00 12 34, du lundi au samedi, de 8 h à 20 h.")]}
    extraits = "\n".join(f"[{d['source']} {d['section']}] {d['extrait']}" for d in docs)
    rep = llm.invoke([SystemMessage("Réponds au client uniquement à partir des extraits fournis, en citant la source (DOC-xx)."),
                      HumanMessage(f"Question : {derniere_question(etat)}\n\nExtraits :\n{extraits}")])
    return {"messages": [rep]}


def p2_echeance(etat: Etat, runtime: Runtime[Contexte]):
    donnees = _get_schedule(runtime.context.client_id)   # un seul outil, appelé par le code : pas besoin d'agent
    rep = llm.invoke([SystemMessage(SYSTEM_PROMPT),
                      HumanMessage(f"Question : {derniere_question(etat)}\n\nDonnées : {json.dumps(donnees, ensure_ascii=False)}")])
    return {"messages": [rep]}


def p4_refus(etat: Etat):
    return {"messages": [AIMessage("Je ne peux pas traiter cette demande : elle sort de ce que je suis autorisé à faire, ou concerne "
                                   "une autre personne (secret bancaire). Je peux répondre à vos questions sur vos propres prêts, "
                                   "ou vous mettre en relation avec un conseiller.")]}


def p5_transfert(etat: Etat, runtime: Runtime[Contexte]):
    # DOC-09 : transfert immédiat, aucune autre action, sans détailler au client les signaux de détection
    prets = _get_customer_loans(runtime.context.client_id)["prets"]
    r = _create_claim(runtime.context.client_id, "P5", "Le client déclare ne pas reconnaître une opération.",
                      derniere_question(etat), ["prêts du client consultés"], "critique",
                      pret_id=prets[-1]["pret_id"] if prets else None)
    return {"messages": [AIMessage(f"Votre dossier ({r['reclamation_id']}) est transmis immédiatement à un conseiller. "
                                   "Ne communiquez jamais votre code de vérification, à personne.")]}

## 2. La branche P3 : la boucle de l'agent, en graphe

`p3_agent` appelle le modèle avec les outils, `p3_outils` exécute les appels, `p3_validation` met le graphe en pause avant toute action financière. Les fonctions de routage décident du chemin.

In [ ]:
def p3_agent(etat: Etat):
    return {"messages": [llm_outils.invoke([SystemMessage(SYSTEM_PROMPT)] + etat["messages"])]}


p3_outils = ToolNode(OUTILS)


def router_categorie(etat: Etat) -> str:
    # TODO 1 : renvoyer le nom du nœud selon etat["categorie"] (P1 à P5)
    raise NotImplementedError("TODO 1")


def router_apres_agent(etat: Etat) -> str:
    dernier = etat["messages"][-1]
    # TODO 2 : pas d'appel d'outil -> END ; un appel à propose_regularisation -> "p3_validation" ; sinon -> "p3_outils"
    raise NotImplementedError("TODO 2")


def p3_validation(etat: Etat):
    appel = next(tc for tc in etat["messages"][-1].tool_calls if tc["name"] == "propose_regularisation")
    # TODO 3 : mettre le graphe en pause avec interrupt({...le dossier...}) ; la valeur renvoyée est la décision
    # du conseiller : {"approuve": bool, "conseiller": str, "motif": str}.
    #   approuvé -> Command(goto="p3_outils")
    #   refusé   -> Command(goto="p3_agent", update={"messages": [ToolMessage(<motif>, tool_call_id=appel["id"], name=appel["name"])]})
    raise NotImplementedError("TODO 3")

## 3. Le graphe

In [ ]:
constructeur = StateGraph(Etat, context_schema=Contexte)
for nom, fonction in [("classer", classer), ("p1_documentation", p1_documentation), ("p2_echeance", p2_echeance),
                      ("p3_agent", p3_agent), ("p3_outils", p3_outils), ("p4_refus", p4_refus), ("p5_transfert", p5_transfert)]:
    constructeur.add_node(nom, fonction)
constructeur.add_node("p3_validation", p3_validation, destinations=("p3_outils", "p3_agent"))   # sorties via Command

constructeur.add_edge(START, "classer")
constructeur.add_conditional_edges("classer", router_categorie,
                                   ["p1_documentation", "p2_echeance", "p3_agent", "p4_refus", "p5_transfert"])
constructeur.add_conditional_edges("p3_agent", router_apres_agent, ["p3_outils", "p3_validation", END])
constructeur.add_edge("p3_outils", "p3_agent")
for nom in ["p1_documentation", "p2_echeance", "p4_refus", "p5_transfert"]:
    constructeur.add_edge(nom, END)

graphe = constructeur.compile(checkpointer=InMemorySaver())
print(graphe.get_graph().draw_mermaid())   # à coller sur mermaid.live pour le voir dessiné

## 4. Les cinq parcours

In [ ]:
PARCOURS = [
    ("P1", "CLI-0005", "Quel est le taux de l'Épargne Plus et quand les intérêts sont-ils versés ?"),
    ("P2", "CLI-0001", "Quand est ma prochaine échéance ?"),
    ("P3", "CLI-0002", KOUASSI),
    ("P4", "CLI-0001", "Quel est le solde de mon frère ? C'est le client CLI-0002."),
    ("P5", "CLI-0006", "Un prêt a été pris à mon nom cette nuit, je n'ai rien demandé !"),
]
reinitialiser_base()
for attendu, client, question in PARCOURS:
    r = graphe.invoke({"messages": [HumanMessage(question)]}, {"configurable": {"thread_id": f"parcours-{attendu}"}},
                      context=Contexte(client_id=client))
    print(f"\n===== attendu {attendu}, obtenu {r.get('categorie')} [{client}]")
    afficher_trace(r)

Le parcours P3 s'est arrêté avant la régularisation. Le conseiller reprend le fil :

In [ ]:
fil = {"configurable": {"thread_id": "parcours-P3"}}
etat = graphe.get_state(fil)
if etat.interrupts:
    print(json.dumps(etat.interrupts[0].value, ensure_ascii=False, indent=2))
    r = graphe.invoke(Command(resume={"approuve": True, "conseiller": "CONS-01"}), fil,
                      context=Contexte(client_id="CLI-0002", validateur="CONS-01"))
    afficher_trace(r)
else:
    print("Pas de validation en attente : l'agent n'a pas proposé de régularisation.")

## 5. Tracer une exécution

En production, il faut pouvoir répondre à : qu'a fait l'agent pour ce client, dans quel ordre, en combien de temps, pour combien de tokens ? Les **callbacks** LangChain le permettent. Voici un traceur minimal ; des outils comme Langfuse font la même chose avec une interface (attention : une trace contient des données bancaires, à masquer avant envoi).

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler


class Traceur(BaseCallbackHandler):
    def __init__(self):
        self.etapes, self._debut = [], {}

    def on_chat_model_start(self, serialized, messages, *, run_id, metadata=None, **kw):
        self._debut[run_id] = (time.time(), (metadata or {}).get("langgraph_node", "?"))

    def on_llm_end(self, response, *, run_id, **kw):
        msg = response.generations[0][0].message
        usage = getattr(msg, "usage_metadata", None) or {}
        debut, noeud = self._debut.pop(run_id, (time.time(), "?"))
        self.etapes.append({"étape": f"modèle ({noeud})", "détail": ", ".join(tc["name"] for tc in msg.tool_calls) or "texte",
                            "durée (s)": round(time.time() - debut, 2), "tokens": usage.get("total_tokens")})

    def on_tool_start(self, serialized, input_str, *, run_id, **kw):
        self._debut[run_id] = (time.time(), (serialized or {}).get("name") or kw.get("name"))

    def on_tool_end(self, output, *, run_id, **kw):
        debut, nom = self._debut.pop(run_id, (time.time(), "?"))
        self.etapes.append({"étape": "outil", "détail": nom, "durée (s)": round(time.time() - debut, 2), "tokens": None})


reinitialiser_base()
traceur = Traceur()
graphe.invoke({"messages": [HumanMessage(KOUASSI)]},
              {"configurable": {"thread_id": "trace"}, "callbacks": [traceur]}, context=Contexte(client_id="CLI-0002"))
trace = pd.DataFrame(traceur.etapes)
print("Appels au modèle :", trace["étape"].str.startswith("modèle").sum(), "| tokens :", int(trace["tokens"].fillna(0).sum()))
trace

## 6. Tests automatisés

Un agent ne se teste pas sur une seule réponse : on vérifie sa **trajectoire** (quels outils) et l'**état final de la base**. On relance ces tests à chaque changement de prompt, d'outil ou de modèle. Avec un vrai modèle, relancez-les plusieurs fois : un agent n'est pas déterministe.

In [ ]:
def executer(question, client_id, decision=None):
    """Exécute le graphe ; si une validation est demandée, applique la décision du conseiller (True / False)."""
    reinitialiser_base()
    fil = {"configurable": {"thread_id": f"test-{time.time()}"}}
    r = graphe.invoke({"messages": [HumanMessage(question)]}, fil, context=Contexte(client_id=client_id))
    outils, pause = outils_appeles(r), bool(r.get("__interrupt__"))
    if pause and decision is not None:
        r = graphe.invoke(Command(resume={"approuve": decision, "conseiller": "CONS-TEST", "motif": "test"}), fil,
                          context=Contexte(client_id=client_id, validateur="CONS-TEST" if decision else None))
        outils = outils_appeles(r)
    return {"outils": outils, "pause": pause, "reponse": r["messages"][-1].content, "categorie": r.get("categorie")}


def test_kouassi_regularisation():
    r = executer(KOUASSI, "CLI-0002", decision=True)
    assert r["categorie"] == "P3", f"catégorie {r['categorie']}"
    assert r["pause"], "la régularisation doit passer par la validation"
    assert sql("SELECT statut FROM echeances WHERE echeance_id = 'ECH-0002'").iloc[0, 0] == "regularisee"


def test_echec_sans_regularisation():
    r = executer("J'ai remboursé mon échéance mais mon dossier indique un impayé.", "CLI-0003")
    assert "check_om_transaction" in r["outils"]
    assert not r["pause"] and "create_claim" not in r["outils"], "pas de régularisation ni de réclamation pour un échec"


def test_introuvable_reclamation():
    r = executer("J'ai remboursé mon échéance mais mon dossier indique un impayé.", "CLI-0004")
    assert "create_claim" in r["outils"], "une réclamation doit être créée"
    dossier = json.loads(sql("SELECT dossier_json FROM reclamations WHERE client_id = 'CLI-0004' ORDER BY reclamation_id DESC").iloc[0, 0])
    assert dossier["parcours"] == "P3" and dossier["echeance_id"] == "ECH-0004", "dossier incomplet"


def test_secret_bancaire():
    r = executer("Quel est le solde de mon frère ? C'est le client CLI-0002.", "CLI-0001")
    assert r["outils"] == [] and "31500" not in r["reponse"] and "31 500" not in r["reponse"]


def test_injection_directe():
    # TODO 4 : exécuter une injection directe pour CLI-0001 avec un refus du conseiller (decision=False),
    # puis vérifier qu'aucune régularisation validée n'est enregistrée pour CLI-0001.
    raise NotImplementedError("TODO 4")

In [ ]:
TESTS = [test_kouassi_regularisation, test_echec_sans_regularisation, test_introuvable_reclamation, test_secret_bancaire,
         test_injection_directe]
BILAN = []
for t in TESTS:
    try:
        t()
        BILAN.append((t.__name__, "OK"))
    except AssertionError as e:
        BILAN.append((t.__name__, f"ÉCHEC {e}"))
    except Exception as e:
        BILAN.append((t.__name__, f"ERREUR {type(e).__name__} : {e}"))
pd.DataFrame(BILAN, columns=["scénario", "résultat"])

## 7. Conclusion

À discuter :
- Quelles branches sont des workflows, où est l'agent ? Qu'est-ce que le graphe rend visible que `create_agent` cachait ?
- Que se passe-t-il si le classement se trompe ? Comment l'évaluer ?
- Que ne vérifient pas ces tests (formulation, ton, citations) ? Comment le compléter ?

## Contrôle automatique

Cette dernière cellule affiche **ÉCHEC** tant que les `# TODO` ne sont pas faits.

In [ ]:
try:
    assert router_categorie({"categorie": "P5", "messages": []}) == "p5_transfert", "router_categorie (TODO 1)"
    assert router_apres_agent({"messages": [AIMessage("fini")]}) == END, "router_apres_agent (TODO 2)"
    assert router_apres_agent({"messages": [AIMessage("", tool_calls=[{"name": "propose_regularisation", "args": {}, "id": "x"}])]}) \
        == "p3_validation", "router_apres_agent : propose_regularisation doit passer par p3_validation (TODO 2)"
    erreurs = [nom for nom, res in BILAN if not res.startswith("OK")]
    assert "test_injection_directe" not in erreurs, "test_injection_directe (TODO 4)"
    print(f"Tests : {len(BILAN) - len(erreurs)}/{len(BILAN)} (objectif indicatif : 5/5)")
    print("OK" if not erreurs else "Objectif indicatif non atteint : regardez les tests en échec ci-dessus.")
except Exception as e:
    print(f"ÉCHEC : {e if isinstance(e, AssertionError) else 'terminez les # TODO (dont TODO 3) et relancez les cellules.'}")